# Try type-safe open-source models on a free Colab GPU
These models answer **typed questions** (yes/no, multiple choice, a score) with probabilities instead of prose. This notebook loads one at a
time on Colab's free **T4 GPU**, opens the **Jev console** (a control panel with a test box and response times), and shows how fast each answers.

**Quick start:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Every cell is safe to run again.

### What will happen
| Step | What it does | Typical time |
|---|---|---|
| 1 · Install | Checks the GPU, disk, network and tools, installs the gateway (included in this file), and installs llama.cpp for the Clef models | 1–2 min |
| 2 · Start | Starts the gateway and opens the Jev console | seconds |
| 3 · Load a model | Pre-flight check (GPU memory, disk, network), then download or install, then start | **2–9 min the first time** (table below); seconds if it is already on disk |
| 4 · Test | Times five calls with text, and an image and a video for models that take them | under a minute |
| 7 · Console | Reopens the Jev console and tells you how to use it | seconds |

### Good to know
* **You will see it working.** Each step prints what it is doing, a line whenever the phase changes, and the elapsed time and the latest log line
  every 30 seconds. If **no new line appears for 5 minutes**, run that cell again: it is safe and picks up where it was.
* **A Hugging Face token makes downloads faster (recommended).** Add it before you run: the key icon in Colab's left sidebar → *Add new secret* →
  name `HF_TOKEN`, value a [read token](https://huggingface.co/settings/tokens) → turn on *Notebook access*. It works without one, just slower.
* **The first model is the slow one.** Besides downloading, the GPU compiles its code once (about a minute). Later models and reloads are quicker.
* **Speed:** an answer comes back in roughly 0.2–1.2 seconds depending on the model. A single T4 serves about 2–3 requests per second; beyond that
  the endpoint answers `429` with a `Retry-After` header. Plenty for trying a model; it is not a production server.
* **One model at a time** is on the GPU. Colab's disk is small, so only the two most recently used models stay downloaded; older ones are deleted
  automatically before a new download.
* **GPU only.** If the GPU can't be used (no GPU, a driver too old, not enough free GPU memory), the notebook stops before downloading anything and says
  why, instead of running slowly on the CPU.

### Models that fit a T4
| Model | What it takes | GPU memory | Download | Typical first-time total |
|---|---|---|---|---|
| **Clef Flash Q4_K_M** (`clef-flash-q4km`) | text, images, video | 10.8 GiB | about 6 GB | 6-9 min |
| **Clef Flash Q2_K** (`clef-flash-q2k`) | text, images, video | 9.3 GiB | about 5 GB | 5-8 min |
| **Laya** (`laya`) | text | 3.1 GiB | about 7 GB | 2-4 min |
| **SemIf** (`semif`) | text | 8.6 GiB | about 13 GB | 3-5 min |
| **open-alternative-jev** (`so1`) | text | 8.6 GiB | about 12 GB | 2-5 min |
| **kev 0.5B** (`kev-0.5b`) | text | 2.7 GiB | about 8 GB | 2-4 min |
| **kev 0.8B** (`kev-0.8b`) | text | 4.5 GiB | about 8 GB | 2-5 min |
| **jeff** (`jeff`) | text | 9.5 GiB | about 8 GB | 2-4 min |

### Included package
The notebook carries the Jev gateway package inside it (the next cell), so nothing is fetched from GitHub to run it. The only downloads are the tools and models listed below.

In [ ]:
#@title The Jev gateway package, embedded in this file (nothing to do here; do not edit)
JEVGW_VERSION = "0.3.2"
JEVGW_SHA256 = "dfea63b6241d878cfa2e1a7fab4cace2211f4c459db383e58ebf195f80f2d318"
JEVGW_WHEEL_B64 = """
...
AAAAADMAMwBJEQAAiMwBAAAA
"""

## 1 · Install and check the machine

In [ ]:
#@title 1 · Install and check the machine (about 1–2 minutes)
import base64, hashlib, shutil, subprocess, sys, tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

def stop(message):
    raise SystemExit(message) from None  # one clear message, no chained traceback

if not shutil.which("nvidia-smi"):
    stop("No GPU found. Runtime > Change runtime type > T4 GPU, then run this cell again.")
home = Path("/content") if Path("/content").is_dir() else Path(tempfile.gettempdir())
marker = home / ".jevgw-package-sha256"
try:
    have = version("jevgw")
except PackageNotFoundError:
    have = None
if have == JEVGW_VERSION and marker.exists() and marker.read_text() == JEVGW_SHA256:
    print(f"The Jev gateway package {JEVGW_VERSION} is already installed.")
else:
    wheel = base64.b64decode(JEVGW_WHEEL_B64)
    if hashlib.sha256(wheel).hexdigest() != JEVGW_SHA256:
        stop("The package inside this notebook is damaged (checksum mismatch). Download the notebook again.")
    print("Installing the Jev gateway package (a few seconds)...")
    path = home / f"jevgw-{JEVGW_VERSION}-py3-none-any.whl"
    path.write_bytes(wheel)
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", str(path)],
                            capture_output=True, text=True)
    if result.returncode:
        stop(f"Could not install the Jev gateway package:\n{result.stderr[-800:]}")
    marker.write_text(JEVGW_SHA256)
    for name in [n for n in sys.modules if n == "jevgw" or n.startswith("jevgw.")]:
        del sys.modules[name]  # use the version just installed, not an older one this session already imported
from jevgw import notebook
LLAMA_BIN = notebook.setup()

## 2 · Start the gateway
The API key is printed below and used by every call. Leave the field empty for a random one, or choose your own.

In [ ]:
#@title 2 · Start the gateway and open the control panel
API_KEY = ""  #@param {type:"string"}
client, info = notebook.start(LLAMA_BIN, key=API_KEY)
KEY = info["key"]
notebook.show_panel(info)

## 3 · Load a model
Pick one and run the cell (or use the panel above). Progress is shown; if a download fails it retries.

In [ ]:
#@title 3 · Load a model
MODEL = "clef-flash-q4km"  #@param ["clef-flash-q4km", "clef-flash-q2k", "laya", "semif", "so1", "kev-0.5b", "kev-0.8b", "jeff"]
notebook.wait_ready(client, MODEL)

## 4 · Test it
Five timed calls with text, then an image and a video's frames for models that take them. Your own files are optional.

In [ ]:
#@title 4 · Test the loaded model
IMAGE = ""  #@param {type:"string"}
VIDEO = ""  #@param {type:"string"}
notebook.test(client, IMAGE, VIDEO)

## 5 · Optional: a public endpoint
Colab's FAQ lists "web service offerings not related to interactive compute" and "connecting to remote proxies" among the things it does not
allow, and says idle runtimes time out. A public URL is the kind of thing those lines describe, so it is **off by default**: use the notebook and
its panel, which is interactive. Turning it on opens a free Cloudflare quick tunnel (testing only: the URL changes if it restarts, and it needs
your API key on every call). Google may disconnect the session or restrict your account; it is your call. The silent-audio keep-alive some people
use is undocumented and may stop working; this notebook does not play it.

In [ ]:
#@title 5 · Public endpoint (off by default)
SERVE = False  #@param {type:"boolean"}
if SERVE:
    tunnel = client.tunnel("start")
    print("endpoint:", tunnel["url"], "   (check it first: open", tunnel["url"] + "/healthz)")
    print("API key :", KEY)
    print(f'curl {tunnel["url"]}/v1/systemone -H "Authorization: Bearer {KEY}" -H "Content-Type: application/json" '
          '-d \'{"state": "Checkout is failing.", "questions": {"outage": {"type": "noul", "instructions": "Is a service down?"}}}\'')
else:
    print("The public endpoint is off.")

## 6 · Stop or reset
Leave on *nothing* for Run all. *reset* also clears a stuck download or load; downloaded models are kept unless you pick the last option.

In [ ]:
#@title 6 · Stop or reset
ACTION = "nothing"  #@param ["nothing", "stop", "reset", "reset and delete downloaded models"]
if ACTION == "stop":
    notebook.stop()
elif ACTION.startswith("reset"):
    notebook.reset(delete_models=ACTION.endswith("models"))

## 7 · Use the Jev console
The last cell reopens the console here and explains what each part does.

In [ ]:
#@title 7 · Open the Jev console and how to use it
notebook.console(client, info)